This notebook is for feature engineering. This is the pipeline for this section:
    
    1. Data gets queried from SQL into this notebook
    2. Data are engineered
    3. Data are arranged in a way that is compatable with the table in SQL
    4. Features are uploaded back into the SQL database

# Database Connection

In [ ]:
# Function
def db_connection(server_name, database_name):
    import pandas as pd
    from sqlalchemy import create_engine

    #server_name = r"DESKTOP-EE25GV9\SQLEXPRESS" # <----- Change This
    #database_name = "stockPredictionApp" # <----- Change This

    # Do not change 'server_name' or 'database_name'
    connection_string = (
        f"mssql+pyodbc://@{server_name}/{database_name}"
        "?driver=ODBC+Driver+17+for+SQL+Server"
        "&trusted_connection=yes"
    )

    engine = create_engine(connection_string) # Connection to Database

    print("Connection successful!")
    
    # Test Query - Find the different tickers
    query = "SELECT * FROM Stocks"

    stocks_df = pd.read_sql(query, engine)

    print("\n",stocks_df)
    return(engine)


# ======================================
# ===== For Azure Database (Cloud) =====

# Not used to protect azure security
def azure_connection():
    
    from sqlalchemy import create_engine
    from urllib.parse import quote_plus

    server = "serverName"
    database = "DLA_StockPrediction"
    username = "username"
    password = "password"

    params = quote_plus(
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"UID={username};"
        f"PWD={password};"
    )

    engine = create_engine(
        f"mssql+pyodbc:///?odbc_connect={params}"
    )

    return(engine)

# Call (Azure)
#azure_connection()


# Call
#engine = db_connection(server_name = r"DESKTOP-EE25GV9\SQLEXPRESS", database_name = "stockPredictionApp")

# Query Data

This query gives all of the closing prices and dates from the stock of your chosing (using stock_id).

In [2]:
def query_data(engine, stock_id):
    import pandas as pd
    
    query = "SELECT PriceDate, ClosePrice, HighPrice, LowPrice, Volume FROM Prices WHERE StockID =" + str(stock_id)
    df = pd.read_sql(query, engine)
    #print(df)
    
    return(df)

# Call
#df = query_data(engine, stock_id = 1)
#print(df)

# Clean Data

- Here we will convert the data from the query into the Features Table

In [3]:
# Use to Refresh Dataframe (For Demos)
#df = query_data(engine, 1)

# Function
def clean_data(df, stock_id, period = 14, window = 20, num_std = 2):
    import numpy as np
    import pandas as pd
    from sqlalchemy import create_engine

    # ===============================================
    # ===== Feature Functions and Calculations =====


    # RSI Calculation
    def calculate_rsi(series, period=14):
        delta = series.diff()
        gain = delta.clip(lower=0)
        loss = -1 * delta.clip(upper=0)

        # Wilder's Smoothing
        avg_gain = gain.ewm(alpha=1 / period, min_periods=period).mean()
        avg_loss = loss.ewm(alpha=1 / period, min_periods=period).mean()

        rs = avg_gain / avg_loss
        return 100 - (100 / (1 + rs))  # Added parentheses around (1 + rs)


    # ATR Calculation
    def calculate_atr(df, period=14):
        high = df["HighPrice"]
        low = df["LowPrice"]
        close = df["ClosePrice"]

        high_low = high - low
        high_prev_close = (high - close.shift(1)).abs()
        low_prev_close = (low - close.shift(1)).abs()

        tr = pd.concat([high_low, high_prev_close, low_prev_close], axis=1).max(axis=1)

        atr = tr.ewm(alpha=1/period, adjust=False).mean()

        return atr


    # ===========================================
    # --- Sample Data Simulation ---
    # (Using mock data that fits both your database columns and test scenario)
    #data = {
    #    "PriceDate": pd.date_range(start="2026-01-01", periods=205, freq="D"),
    #    "HighPrice": np.random.uniform(100, 110, 205),
    #    "LowPrice": np.random.uniform(90, 100, 205),
    #    "ClosePrice": np.random.uniform(95, 105, 205),
    #    "Volume": np.random.randint(1000, 5000, 205),
    #}
    #df = pd.DataFrame(data)

    # ===========================================
    # ---------- Main Processing Pipeline ----------

    # Bollinger Bands calculations

    df["BB_Mid"] = df["ClosePrice"].rolling(window=window).mean()
    df["BB_Std"] = df["ClosePrice"].rolling(window=window).std(ddof=0)

    # Apply indicators and features
    df["StockID"] = stock_id
    df["FeatureDate"] = df["PriceDate"]
    df["SMA_50"] = df["ClosePrice"].rolling(window=50).mean()
    df["SMA_200"] = df["ClosePrice"].rolling(window=200).mean()
    df["RollingVariance_50"] = df["ClosePrice"].rolling(window=50).var()
    df["RSI"] = calculate_rsi(series=df["ClosePrice"])
    df["BollingerUpper"] = df["BB_Mid"] + (num_std * df["BB_Std"])
    df["BollingerLower"] = df["BB_Mid"] - (num_std * df["BB_Std"])
    df["ATR"] = calculate_atr(df)
    df["VolumeChange"] = df["Volume"].diff()

    df_ret = df.iloc[:,7:]
    # Display or return result
    print(df_ret.tail())
    return(df_ret) # <------------------------- Send This to Database


#features = clean_data(df, stock_id = 1)

# Send Features Column to SQL Database

In [4]:
# Function
def insert_sql(df, engine, stock_id):
    import pandas as pd
    from sqlalchemy import create_engine
    import yfinance as yf
    
    if stock_id == 1:
        df.to_sql(
        "Features",
        engine,
        if_exists="replace",
        index=False
    )
    else:
        df.to_sql(
        "Features",
        engine,
        if_exists="append",
        index=False
    )
    

    print("Features inserted successfully!")
    
    # Sample Query
    query = "SELECT TOP 10* FROM Features WHERE StockID = " + str(stock_id)
    df = pd.read_sql(query, engine)

    print(df.head())

#insert_sql(bac_features, engine, stock_id = 1)

# Full Feature Engineering Pipeline

- With this code block, you can turn a stock data dataframe into a features table in SQL. 

In [7]:
def feature_engineering(s_id): # s_id = Stock ID
    
    # 1. Connect to the Database
    engine = db_connection(server_name = r"DESKTOP-EE25GV9\SQLEXPRESS", database_name = "stockPredictionApp")
    #engine = azure_connection() # <--------------- for Azure (Cloud) Database
    
    # 2. Query Price Data - Checks the Connection
    df = query_data(engine, stock_id = s_id)
    print(df.head())

    # 3. Clean Data - Feature Engineering
    features = clean_data(df, stock_id = s_id)

    # 4. Send to Database
    insert_sql(features, engine, stock_id = s_id)
    
    print("Features Added!")


feature_engineering(s_id = 3) # <----------------------------- Change the Stock ID

Connection successful!

    StockID Ticker       CompanyName      Sector
0        1    BAC   Bank of America  Financials
1        2    TFC  Truist Financial  Financials
2        3    WFC       Wells Fargo  Financials
   PriceDate  ClosePrice  HighPrice   LowPrice    Volume
0 2020-01-02   45.566654  45.812503  45.363192  16803100
1 2020-01-03   45.286896  45.456447  44.846068  15608800
2 2020-01-06   45.015614  45.100391  44.693470  13200300
3 2020-01-07   44.642597  44.973220  44.481526  13278600
4 2020-01-08   44.778244  45.210599  44.761289  16585600
      StockID FeatureDate     SMA_50    SMA_200  RollingVariance_50   
1631        3  2026-07-01  79.768927  83.651719           12.342388  \
1632        3  2026-07-02  79.857399  83.678619           12.971628   
1633        3  2026-07-06  80.003960  83.715317           14.124762   
1634        3  2026-07-07  80.146513  83.745301           15.154908   
1635        3  2026-07-08  80.296543  83.766715           15.919671   

            RS